### MNIST Binary Classifier

In this notebook, we will implement a DNN classifier to classify the digits 0 and 1 from the MNIST dataset. The full classifier (for all digits) we will implement in the next lesson. The objective of this lesson is twofold:
* To build our first DNN classifier (binary).
* To demonstrate the importance of data normalization.

Let's start with the usual imports.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

from time import time
import numpy as np
import keras
from keras import Model
from keras.layers import Input, Dense, Flatten

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

#### Dataset Loading

We have already inspected the MNIST dataset. We are going to load it now since we are going to use it for training the classifier.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Dataset params
num_classes = 10
size = x_train.shape[1]

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

#### Dataset Preprocessing

In this example, we are going to train a binary classifier to classify the digits 0 and 1. Therefore, we have to remove all other digits (classes) from the dataset.

In [ ]:
mask_train = np.logical_or(y_train == 0, y_train == 1)
x_train = x_train[mask_train, ...]
y_train = y_train[mask_train]

mask_test = np.logical_or(y_test == 0, y_test == 1)
x_test = x_test[mask_test, ...]
y_test = y_test[mask_test]

print('Train set:   ', len(y_train), 'samples')
print('Test set:    ', len(y_test), 'samples')
print('Sample dims: ', x_train.shape)

#### Building the Classifier

We are going to build a relatively simple fully-connected DNN for this task. Since we will need to build the very same network more than once (to start each training from fresh random weights), we wrap it into a function.

In [ ]:
def build_model():
    inputs = Input(shape=(size, size, 1))

    net = Flatten()(inputs)
    net = Dense(16, activation='relu')(net)
    outputs = Dense(1)(net)

    return Model(inputs, outputs)

model = build_model()
model.summary()

This is an extremely simple model (for a usual classification task) yet it already contains several thousand of (trainable) parameters.

#### Training

Let's now compile and train the model. We will use the well-known MSE as our loss function.

Note: MSE is **not** the suitable loss for classification task but it serves us here well for the demonstration purposes. We will learn how to design a classifier in a proper way in the next lesson ;-)

Also note that, **on purpose**, we feed the images to the network exactly as they are, i.e., with pixel values in [0, 255]. We will see in a moment why this is not a good idea.

In [ ]:
epochs = 25
batch_size = 128

model.compile(loss="mse", optimizer="adam", metrics=["accuracy"])

start = time()
history_raw = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=2)
print('Elapsed time', time() - start)

Let's now plot the history to see the evolution of the training.

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])

    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])

plot_history(history_raw)

#### Evaluation

From the history we see that the training performance is quite consistent with the validation (which is good, we will learn about the overfitting problem in next lessons). Now we are going to evaluate the trained classifier on the test dataset. Remember, this is the dataset that the network has **not** seen during the training and it will be used to assess the final performance of the model.

The network outputs a real number. As in the previous lesson, we turn it into a class by thresholding at 0.5. We wrap the evaluation into functions, since we will repeat it later.

In [ ]:
def evaluate(model, x, y):
    y_score = model.predict(x, verbose=0).flatten()
    y_pred = y_score > 0.5

    print('Overall acc', np.mean(y_pred == y))
    print('Digit-0 acc', np.mean(y_pred[y == 0] == y[y == 0]))
    print('Digit-1 acc', np.mean(y_pred[y == 1] == y[y == 1]))

    return y_score

def show_predictions(x, y, y_score):
    for ii in range(15):
        idx = np.random.randint(0, len(y))
        plt.subplot(3,5,ii+1), plt.imshow(x[idx, ...], cmap='gray'), plt.axis(False)
        plt.title('True: ' + str(y[idx]) + ' | Pred: ' + str(int(y_score[idx] > 0.5)) +
                  ' (score ' + format(y_score[idx], '.2f') + ')')

y_score = evaluate(model, x_test, y_test)

In [ ]:
show_predictions(x_test, y_test, y_score)

### Data Normalization

The dynamic range of the input signals (images) is [0, 255] and the (groundtruth) output lies within the interval [0, 1]. This is a huge disproportion between the input and output ranges and the network needs to learn to compensate for that. This will slow the convergence and, in general, yields poorer results. In order to reduce the negative effect of the input-output range mismatch, data **normalization** is necessary. The objective of data normalization is to harmonize the input and output ranges and to let the network to focus on learning the important classification features instead of learning also the range compensation factors.

Normalizing is as simple as dividing the images by 255, so that they fall into [0, 1].

In [ ]:
# Data normalization
x_train = x_train/255
x_test = x_test/255

Let's train the very same network again, from new random weights, but now on the normalized data.

In [ ]:
model = build_model()

model.compile(loss="mse", optimizer="adam", metrics=["accuracy"])
start = time()
history_norm = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=0)
print('Elapsed time', time() - start)

plot_history(history_norm)

In [ ]:
y_score = evaluate(model, x_test, y_test)
show_predictions(x_test, y_test, y_score)

To see the effect of the normalization clearly, let's put the two training histories into the same plot. Notice that the loss axis is logarithmic: the loss of the raw-input network starts at a value of the order of thousands, since its initial outputs are totally off the [0, 1] range of the labels. The normalized network starts several orders of magnitude lower.

In [ ]:
def compare_histories(histories):
    for name, history in histories.items():
        h = history.history
        plt.subplot(121), plt.semilogy(h['loss'], '.-', label=name)
        plt.subplot(122), plt.plot(h['val_accuracy'], '.-', label=name)
    plt.subplot(121), plt.grid(True), plt.xlabel('epochs'), plt.ylabel('train loss'), plt.legend()
    plt.subplot(122), plt.grid(True), plt.xlabel('epochs'), plt.ylabel('validation accuracy'), plt.legend()

compare_histories({'raw [0, 255]': history_raw, 'normalized [0, 1]': history_norm})

### Data Standardization

A step further is the data **standardization**: we subtract the mean and divide by the standard deviation, so that the data have zero mean and unit standard deviation,

$$x' = \frac{x - \mu}{\sigma}$$

Important: the mean and the standard deviation are computed on the **training** set only, and exactly the same values are then used for the test set (at test time, the model must not know anything about the test data).

Here we compute $\mu$ and $\sigma$ over the whole training set. One can also standardize each image on its own, which makes the network insensitive to the image brightness and contrast. That is useful when the illumination is a nuisance, but harmful when the brightness is exactly the signal we want to classify (as in day vs night, see next notebook).

In [ ]:
mean, std = np.mean(x_train), np.std(x_train)
print('Train mean', mean, '| Train std', std)

x_train_std = (x_train - mean)/std
x_test_std = (x_test - mean)/std

model = build_model()
model.compile(loss="mse", optimizer="adam", metrics=["accuracy"])
history_std = model.fit(x_train_std, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=0)

plot_history(history_std)
y_score = evaluate(model, x_test_std, y_test)

In [ ]:
compare_histories({'raw [0, 255]': history_raw, 'normalized [0, 1]': history_norm, 'standardized': history_std})

MNIST is an easy dataset and the normalization alone already does most of the job; standardization does not bring much more here. It will matter more for harder data, with deeper networks and with images of varying brightness and contrast.